# 1. Library calling

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import datetime
import warnings

# Ignore all warnings (not recommended in general)
warnings.filterwarnings("ignore")

# 2. Defining the Product Information and Location

In [2]:
SummaryFolder=r'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects'
summaryFile='Scraping_List.txt'
st=pd.read_csv(SummaryFolder+'\\'+summaryFile)
print(st)
#Define Product to extract
search_text = st['Product Name'][9]
print(search_text)
Source="HomeDepo"
OFolder=fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\{Source}\Outputs'
IFolder=fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\{Source}\Inputs'
filename=Source+'ProductLinks_'+search_text+'.xlsx'
df1=pd.read_excel(IFolder+'\\'+filename)
df1


                                  Product Name
0                                Ignition Coil
1                      Windshield Washer Pumps
2                        coupler trailer locks
3          Adjustable Trailer Hitch Ball Mount
4                               Vacuum Cleaner
5                                   Spark Plug
6              bluetooth Enabled Trailer locks
7                          Power Steering Hose
8   Power Steering Pressure Line Hose Assembly
9                       Washer Fluid Reservoir
10          Hitch Ball Mount with Weight Scale
11                               LED Headlamps
12                             LED flashlights
13                   Fiberglass Tonneau Covers
14                    Aluminium Tonneau Covers
15                     Hardfold Tonneau Covers
16                            Spark Plug Wires
17                 washer fluid reservoir tank
18                   Non Automotive Gas Struts
Washer Fluid Reservoir


,Links,Name
0,https://www.homedepot.com/p/OE-Solutions-Winds...,OE Solutions\nWindshield Washer Fluid Reservoir
1,https://www.homedepot.com/p/Washer-Fluid-Reser...,Washer Fluid Reservoir Cap 2012-2017 Ford Focu...
2,https://www.homedepot.com/p/OE-Solutions-Winds...,OE Solutions\nWindshield Washer Fluid Reservoir
3,https://www.homedepot.com/p/Rain-X-128-fl-oz-3...,Rain-X\n128 fl. oz. +32°F Degree Bug Remover W...
4,https://www.homedepot.com/p/Washer-Fluid-Reser...,Washer Fluid Reservoir Cap
...,...,...
109,https://www.homedepot.com/p/Thinkcar-8-in-OBD2...,Thinkcar\n8 in. OBD2 Scanner Tablet Profession...
110,https://www.homedepot.com/p/Thinkcar-OBD2-Scan...,Thinkcar\nOBD2 Scanner Car Code Reader Check E...
111,https://www.homedepot.com/p/Thinkcar-Professio...,Thinkcar\nProfessional Mechanic OBD2 Scanner C...
112,https://www.homedepot.com/p/VEVOR-Refrigerant-...,NaN


In [3]:
df1.loc[1,"Name"]

'Washer Fluid Reservoir Cap 2012-2017 Ford Focus 2.0L'

In [4]:
links=[]
for i in range(len(df1)):
    # if " " in df1.loc[i,"Name"]:
        links.append(df1['Links'][i])

In [5]:
length=len(links)

# 3. Setting Webdriver and Website Specific Information

In [15]:
path= 'C://chromedriver.exe'
driver=webdriver.Chrome()
driver.get('https://www.homedepot.com/')
driver.maximize_window()

wait=WebDriverWait(driver, 5)

In [16]:
location = driver.find_element(By.CSS_SELECTOR, '[data-testid="delivery-zip-button"]')
location.click()
sleep(1)
pin = driver.find_element(By.CSS_SELECTOR, '[placeholder="Enter ZIP Code"]')
pinnumber='94203'
pin.send_keys(pinnumber)
pin.submit()

# 4. Defining the Dataframe and Extracting the data into the Dataframe

In [17]:
cols =['Sl.No','Attributes'] 
df = pd.DataFrame(columns=cols)
df
count=0

In [9]:
from IPython.display import clear_output 
import numpy as np
from datetime import timedelta
import datetime 
timestamp=[]
timediff=[]
def timeremaining(balanceitem,i):
    timestamp.insert(i,datetime.datetime.now())
    if i>=1:
        diff=timestamp[i]-timestamp[i-1]
        timediff.insert(i,diff.total_seconds())		
        remainingtime=np.median(timediff)*balanceitem
        millis=int(remainingtime)		
        #print(millis)
        seconds=(millis)%60
        seconds = int(seconds)
        minutes=(millis/(60))%60
        minutes = int((minutes)) #math.floor
        hours=(millis/(60*60))%24
        hours=int(hours)
        print(f'Item looped: {i+1} \nItems Remining: {balanceitem-1} and \nTime Remaining: {hours}:{minutes}:{seconds}')
        etc=datetime.datetime.now()+timedelta(hours=hours,minutes=minutes,seconds=seconds)
        print(f'Estimated time of completion:{etc}')        
        clear_output(wait=True)

In [22]:
from tqdm import tqdm
for i in tqdm(range(57,length)):
    driver.get(links[i])
    sleep(1)
    df.loc[count,'Sl.No']=i
    df.loc[count,'Name']=driver.find_element(By.CSS_SELECTOR, '[data-component="ProductDetailsTitle"]').text
    df.loc[count,'Current Price']=float(driver.find_element(By.ID, 'standard-price').text.replace('''\n''',"").replace('''$''',"").replace('''/box''',"").split(" ")[0])
    driver.find_element(By.CSS_SELECTOR,'[id="product-details__review__target"]').click()
    sleep(2)
    try:
        df.loc[count,'Rating']=driver.find_element(By.CSS_SELECTOR,'[class="sui-text-center"]').text.split(' out')[0]
    except:
        df.loc[count,'Rating']=0
    try:
        df.loc[count,'No of Ratings']=driver.find_element(By.CSS_SELECTOR,'[class="sui-font-regular sui-text-xs sui-leading-tight sui-tracking-normal sui-normal-case sui-line-clamp-unset sui-text-primary"]').text
    except:
        df.loc[count,'No of Ratings']=0
    sleep(1)
    elements=driver.find_elements(By.CLASS_NAME,'sui-mr-2')
    Alist=[]
    for element in elements:
        if "#" in element.text:
            Alist.append(element.text.replace(" # ",":"))
    driver.execute_script("window.scrollTo(0, 1500);")   
    sleep(1) 
    spbutton=wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '[class="navlink-specs"]')))
    spbutton.click()
    try:
        Aelements=wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME,"specifications__cell__label")))
        Velements=driver.find_elements(By.CSS_SELECTOR,'[class="specifications__cell"]')        
        for a, v in zip(Aelements, Velements):
            Alist.append(a.text+":"+v.text)
        df.at[count,'Attributes']=Alist
    except:
        pass    
    pdbutton=driver.find_element(By.CSS_SELECTOR,'[class="navlink-pso"]').click()
    try:
        PDetail=wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '[class="grid desktop-content-wrapper__main-description"]'))).text            
        df.loc[count,'Details']=PDetail
    except: 
        pass
    df.loc[count,'Links']=links[i]
    df.loc[count,'Source']="HomeDepot"
    df.loc[count,'Product']=search_text
    count=count+1
    balanceitem=length-i
    #print(balanceitem)
    #clear_output(wait=True)
    #timeremaining(balanceitem,i)
    sleep(2)

100%|██████████| 57/57 [18:00<00:00, 18.95s/it]


In [23]:
print(df.shape)
df.head()

(114, 9)


,Sl.No,Attributes,Name,Current Price,Rating,No of Ratings,Links,Source,Product
0,0,"[Internet:307879020, Model:603-117, Store SKU:...",Windshield Washer Fluid Reservoir,19.18,5,(1),https://www.homedepot.com/p/OE-Solutions-Winds...,HomeDepot,Washer Fluid Reservoir
1,1,"[Internet:311347767, Model:54009, Store SKU:10...",Washer Fluid Reservoir Cap 2012-2017 Ford Focu...,10.08,4,(8),https://www.homedepot.com/p/Washer-Fluid-Reser...,HomeDepot,Washer Fluid Reservoir
2,2,"[Internet:307969654, Model:603-173, Store SKU:...",Windshield Washer Fluid Reservoir,27.12,0,0,https://www.homedepot.com/p/OE-Solutions-Winds...,HomeDepot,Washer Fluid Reservoir
3,3,"[Internet:204981587, Model:113605, Store SKU:1...",128 fl. oz. +32°F Degree Bug Remover Windshiel...,4.57,4.7,(206),https://www.homedepot.com/p/Rain-X-128-fl-oz-3...,HomeDepot,Washer Fluid Reservoir
4,4,"[Internet:311983067, Model:54117, Store SKU:10...",Washer Fluid Reservoir Cap,3.73,0,0,https://www.homedepot.com/p/Washer-Fluid-Reser...,HomeDepot,Washer Fluid Reservoir


# 5. Post Processing Data and Exporting

In [24]:
df['Sl.No']=df['Sl.No'].astype(int)
df['No of Ratings']=df['No of Ratings'].str.replace(")",'').str.replace("(",'').astype(float)
df['Rating']=df['Rating'].astype(float)

In [25]:
df['Part Number']=df['Attributes'].apply(lambda x: str(x).replace('[','').replace(']','').replace('\'', '').replace(' ','')).str.split('Model:',expand=True)[1].str.split(',',expand=True)[0]

In [27]:
cols=["Sl.No",
"Name",
"Product",
"Current Price",
"Rating",
"No of Ratings",
"Attributes",
"Part Number",
"Links",
"Source"
]
df=df[cols]
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 114 entries, 0 to 113
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Sl.No          114 non-null    int32  
 1   Name           114 non-null    object 
 2   Product        114 non-null    object 
 3   Current Price  114 non-null    float64
 4   Rating         114 non-null    float64
 5   No of Ratings  32 non-null     float64
 6   Attributes     114 non-null    object 
 7   Part Number    114 non-null    object 
 8   Links          114 non-null    object 
 9   Source         114 non-null    object 
dtypes: float64(3), int32(1), object(6)
memory usage: 13.4+ KB


In [28]:
df

,Sl.No,Name,Product,Current Price,Rating,No of Ratings,Attributes,Part Number,Links,Source
0,0,Windshield Washer Fluid Reservoir,Washer Fluid Reservoir,19.18,5.0,1.0,"[Internet:307879020, Model:603-117, Store SKU:...",603-117,https://www.homedepot.com/p/OE-Solutions-Winds...,HomeDepot
1,1,Washer Fluid Reservoir Cap 2012-2017 Ford Focu...,Washer Fluid Reservoir,10.08,4.0,8.0,"[Internet:311347767, Model:54009, Store SKU:10...",54009,https://www.homedepot.com/p/Washer-Fluid-Reser...,HomeDepot
2,2,Windshield Washer Fluid Reservoir,Washer Fluid Reservoir,27.12,0.0,NaN,"[Internet:307969654, Model:603-173, Store SKU:...",603-173,https://www.homedepot.com/p/OE-Solutions-Winds...,HomeDepot
3,3,128 fl. oz. +32°F Degree Bug Remover Windshiel...,Washer Fluid Reservoir,4.57,4.7,206.0,"[Internet:204981587, Model:113605, Store SKU:1...",113605,https://www.homedepot.com/p/Rain-X-128-fl-oz-3...,HomeDepot
4,4,Washer Fluid Reservoir Cap,Washer Fluid Reservoir,3.73,0.0,NaN,"[Internet:311983067, Model:54117, Store SKU:10...",54117,https://www.homedepot.com/p/Washer-Fluid-Reser...,HomeDepot
...,...,...,...,...,...,...,...,...,...,...
109,109,8 in. OBD2 Scanner Tablet Professional Vehicle...,Washer Fluid Reservoir,969.95,5.0,1.0,"[Internet:321434387, Model:TKT03, Store SKU:10...",TKT03,https://www.homedepot.com/p/Thinkcar-8-in-OBD2...,HomeDepot
110,110,OBD2 Scanner Car Code Reader Check Engine Ligh...,Washer Fluid Reservoir,58.15,4.5,6.0,"[Internet:321455565, Model:THINKOBD500, Store ...",THINKOBD500,https://www.homedepot.com/p/Thinkcar-OBD2-Scan...,HomeDepot
111,111,Professional Mechanic OBD2 Scanner Car Code Re...,Washer Fluid Reservoir,4699.95,0.0,NaN,"[Internet:326814942, Model:301050008, Store SK...",301050008,https://www.homedepot.com/p/Thinkcar-Professio...,HomeDepot
112,112,Refrigerant Recovery Tank 30 lbs. Capacity Ref...,Washer Fluid Reservoir,72.30,5.0,3.0,"[Internet:323447780, Model:LMHSJ30LBZLJG0001V0...",LMHSJ30LBZLJG0001V0,https://www.homedepot.com/p/VEVOR-Refrigerant-...,HomeDepot


In [29]:
dfAtt=df[['Part Number','Attributes']]
dfAtt=dfAtt.explode('Attributes')
dfAtt[['Attributes', 'Value']] = dfAtt['Attributes'].str.split(':',1, expand=True)

In [30]:
summary_df = pd.pivot_table(dfAtt, values='Value', index='Attributes',aggfunc='count').reset_index()
summary_df =summary_df.sort_values(by='Value',ascending=False)
summary_df=summary_df.rename(columns ={'Value':'No Products contains this Attribute'})
summary_df

,Attributes,No Products contains this Attribute
22,Model,114
18,Internet,114
35,Returnable,112
39,Store SKU,107
3,Automotive Part Type,69
16,General Part Type,44
31,Product Weight (lb.),40
23,Number of Pieces Included,36
21,Material,35
37,Shop Equipment Product Type,32


In [31]:
with pd.ExcelWriter(OFolder+'\\'+f'{Source}ProductDetails_'+search_text+'.xlsx') as writer:  # doctest: +SKIP
    df.to_excel(writer,index=False, sheet_name='Raw')
    summary_df.to_excel(writer,index=False, sheet_name='Attribute_Summary')

# 99. Archived Codes

In [ ]:
# for i in range(42,127):#len(links)):
#     Rows=[]
#     driver_new.get(links[i])
#     Rows.append(i)
#     sleep(1)
#     product = driver_new.find_elements(By.XPATH, "//span[@class='a-size-large product-title-word-break']")
#     if len(product)==0:
#         Rows.append("-")
#     else:
#         for n in product:
#             Rows.append(n.text)
#     price=driver_new.find_elements(By.XPATH, "//span[@class='a-price a-text-price a-size-medium apexPriceToPay']")
#     if len(price)==0:
#         price=driver_new.find_elements(By.XPATH, "//span[@class='a-price aok-align-center reinventPricePriceToPayMargin priceToPay']")
#     if len(price)==0:
#         Rows.append("-")
#     else:
#         for pr in price:
#             if "\n" in pr.text:
#                 prm=pr.text.replace('\n',".")
#                 Rows.append(prm)
#             else:    
#                 if pr.text== "":
#                     pass
#                 else: 
#                     Rows.append(pr.text)
#     try:
#         listprice=driver_new.find_element(By.CLASS_NAME,"basisPrice")
#         Rows.append(listprice.text.split("$")[1])
#     except:
#         for pr in price:
#             if "\n" in pr.text:
#                 prm=pr.text.replace('\n',".")
#                 Rows.append(prm)
#             else:    
#                 if pr.text== "":
#                     pass
#                 else: 
#                     Rows.append(pr.text)
#     try:
#         rating=driver_new.find_element(By.ID, "averageCustomerReviews")
#         if len(rating.text)==0:
#             Rows.append("-")
#         else:
#             Rows.append(rating.text)
#     except:
#         Rows.append("-")
#     PDetail=driver_new.find_elements(By.ID, "productDetails_techSpec_section_1")
#     if len(PDetail)==0:
#         Rows.append("-") 
#     else:
#         for det in PDetail:
#             Rows.append(det.text)
#     Sales = driver_new.find_elements(By.XPATH, "(//span[@class='a-size-small social-proofing-faceout-title-text'])")
#     if len(Sales)==0:
#         Rows.append("-")
#     else:
#         for s in Sales:
#             Rows.append(s.text)
#     IDetails=driver_new.find_elements(By.XPATH, "(//div[@class='a-section a-spacing-medium a-spacing-top-small'])")
#     if len(IDetails)==0:
#         Rows.append("-") 
#     else:
#         for Idet in IDetails:
#             Rows.append(Idet.text)
#     #driver_new.find_element(By.TAG_NAME,'body').send_keys(Keys.END)
#     #sleep(3)
#     try:
#         crt=driver_new.find_element(By.ID,'acrCustomerReviewText')
#         crt.click()
#     except:
#         print("no Review")
#     sleep(2)
#     # try:
#     #     Reveiews=driver_new.find_element(By.ID, "cr-lighthut-1-")
#     #     list_items = Reveiews.find_elements(By.CLASS_NAME,"cr-lighthouse-term ")
#     #     ListWords=[]
#     #     for item in list_items:
#     #         ListWords.append(item.text)
#     #     Rows.append(ListWords)         
#     # except:
#     #     Rows.append("-") 
#     Rows.append(links[i])
#     new_row=pd.DataFrame([Rows],columns=cols)
#     df=pd.concat([df,new_row])
#     print("Scanned pages:", i+1,"@",datetime.datetime.now())

In [ ]:
def tryconvert(value, default, *types):
    for t in types:
        try:
            return t(value)
        except (ValueError, TypeError):
            continue
    return default